# Weak-label exploration v1 — urgency on customer tickets

Notebook-first exploratory slice. The goal is a **first, defensible** weak label for ticket urgency on `inbound = True` customer tweets from `data/raw/twcs.csv`.

Scope of this notebook:
1. Load raw CSV
2. Select the 7 relevant columns
3. Base text cleaning
4. Filter to `inbound = True`
5. Define a simple urgent / normal weak-label rule
6. Apply it, inspect class distribution + examples
7. Save `data/processed/customer_tickets_weak_labeled_v1.csv`

Out of scope: engineered features, training, embeddings, pipeline integration.

## 1. Imports and paths

In [1]:
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_CSV       = PROJECT_ROOT / "data" / "raw" / "twcs.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Optional cap for quick iteration. Set to None for the full dataset.
NROWS = None

RELEVANT_COLUMNS = [
    "tweet_id",
    "author_id",
    "inbound",
    "created_at",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id",
]
RAW_CSV, PROCESSED_DIR, NROWS

(WindowsPath('C:/Users/Jamal/Documents/bootcamps/AIE/Week 3/Decision-Intelligence-Assistant/data/raw/twcs.csv'),
 WindowsPath('C:/Users/Jamal/Documents/bootcamps/AIE/Week 3/Decision-Intelligence-Assistant/data/processed'),
 None)

## 2. Load + select columns + base cleaning

Base cleaning mirrors the pipeline: drop null text, cast to string, strip, drop empties, coerce `inbound` to real bool.

In [2]:
df = pd.read_csv(RAW_CSV, nrows=NROWS)
print("raw rows:", {df.shape})
df = df[RELEVANT_COLUMNS].copy()

df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()

if df["inbound"].dtype != bool:
    df["inbound"] = df["inbound"].map({True: True, False: False, "True": True, "False": False})
    df = df.dropna(subset=["inbound"]).copy()
    df["inbound"] = df["inbound"].astype(bool)

print("rows after base cleaning:", {df.shape})
df.head(2)

raw rows: {(2811774, 7)}


rows after base cleaning: {(2811774, 7)}


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0


## 3. Filter to support tickets (`inbound = False`)

In [3]:
support = df[~df["inbound"]].copy()
print("support tickets:", len(support))
support.head(2)

support tickets: 1273931


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0


## 4. Filter to customer tickets (`inbound = True`)

In [4]:
customer = df[df["inbound"]].copy()
print("customer tickets:", len(customer))
customer.head(2)

customer tickets: 1537843


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0


## 5. Weak-label rule v1

**Why this rule.** We want a *first* signal that is cheap, transparent, and defensible in review. A keyword + punctuation heuristic is the simplest thing that could work:

- **Keyword groups** target the language customers use when something is blocking or financial: `refund / charged / billing / payment / overcharged`, broken-service wording (`broken / not working / down / error / issue / problem`), access loss (`locked / can't log in / access`), explicit urgency (`urgent / asap / immediately / help`), and escalation signals (`cancel / missing / delayed`).
- **Punctuation intensity** — repeated `!!` or `???` — captures affect that isn't keyword-based.
- Word-boundary anchors (`\b`) reduce false positives like `downtown` matching `down`.
- Case-insensitive.

Known limits: this is a **binary proxy**, not ground truth. It will over-call some support tweets (false positives on `help` in casual use) and miss subtle urgency. A later slice will compare against a held-out human-labeled sample.

In [5]:
URGENT_KEYWORD_PATTERNS = [
    r"\brefund(ed|s)?\b",
    r"\bcharged?\b",
    r"\bbilling\b",
    r"\bpayment(s)?\b",
    r"\bovercharged?\b",
    r"\bbroken\b",
    r"\bnot working\b",
    r"\bdown\b",
    r"\berror(s)?\b",
    r"\bissue(s)?\b",
    r"\bproblem(s)?\b",
    r"\block(ed)?\b",
    r"\bcan'?t log ?in\b",
    r"\bcannot log ?in\b",
    r"\baccess\b",
    r"\burgent\b",
    r"\basap\b",
    r"\bimmediately\b",
    r"\bhelp\b",
    r"\bcancel(lation|led|s)?\b",
    r"\bmissing\b",
    r"\bdelayed\b",
]
URGENT_REGEX = re.compile("|".join(URGENT_KEYWORD_PATTERNS), re.IGNORECASE)
PUNCT_REGEX  = re.compile(r"!{2,}|\?{2,}")

def weak_label_urgency(text: str) -> str:
    if URGENT_REGEX.search(text) or PUNCT_REGEX.search(text):
        return "urgent"
    return "normal"
# Sanity check
for t in [
    "my account is locked and I can't log in",
    "please refund my payment ASAP",
    "great service today",
    "help!!! my flight is delayed",
]:
    print(f"{weak_label_urgency(t):>6}  |  {t}")

urgent  |  my account is locked and I can't log in
urgent  |  please refund my payment ASAP
normal  |  great service today
urgent  |  help!!! my flight is delayed


## 6. Apply label

In [6]:
customer["priority_label"] = customer["text"].map(weak_label_urgency)
customer["priority_label"].value_counts(dropna=False)

priority_label
normal    1167203
urgent     370640
Name: count, dtype: int64

## 7. Distribution + example inspection

In [7]:
dist = customer["priority_label"].value_counts(normalize=True).round(4)
print("class distribution (proportion):")
print(dist)
print()
print("absolute counts:")
print(customer["priority_label"].value_counts())

class distribution (proportion):
priority_label
normal    0.759
urgent    0.241
Name: proportion, dtype: float64

absolute counts:
priority_label
normal    1167203
urgent     370640
Name: count, dtype: int64


In [8]:
print("--- urgent examples ---")
for t in customer.loc[customer["priority_label"] == "urgent", "text"].head(5).tolist():
    print("-", t[:180])
print()
print("--- normal examples ---")
for t in customer.loc[customer["priority_label"] == "normal", "text"].head(5).tolist():
    print("-", t[:180])

--- urgent examples ---
- actually that's a broken link you sent me and incorrect information https://t.co/V4yfrHR8VI
- somebody from @VerizonSupport please help meeeeee 😩😩😩😩 I'm having the worst luck with your customer service
- @VerizonSupport What else can I provide? They refuse to help me because they cannot validate the account...
- .@VerizonSupport @115725 @115726                                                 &gt;All of VERIZON IS DOWN&lt;
When can we expect a fix ?
- @ChipotleTweets Thank you @ChipotleTweets for resolving my issue so quickly!! Y’all are the best ☺️ #fanforlife

--- normal examples ---
- @sprintcare and how do you propose we do that
- @sprintcare I have sent several private messages and no one is responding as usual
- @sprintcare I did.
- @sprintcare is the worst customer service
- @sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯


## 8. Save labeled CSV to `data/processed/`

In [9]:
out_path = PROCESSED_DIR / "customer_tickets_weak_labeled_v1.csv"
customer.to_csv(out_path, index=False)
print("wrote:", out_path)
print("rows:", len(customer))
print("columns:", list(customer.columns))

wrote: C:\Users\Jamal\Documents\bootcamps\AIE\Week 3\Decision-Intelligence-Assistant\data\processed\customer_tickets_weak_labeled_v1.csv
rows: 1537843
columns: ['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'priority_label']


## 9. EDA: leading `@`-mention prefix pattern

Twitter customer-support tweets are conversational replies, so they often start with an `@` token addressing someone. Two distinct shapes are visible in the data:

- Customer tweets tend to begin with `@CompanyHandle` (the brand they're complaining to).
- Support replies tend to begin with `@123456` — an anonymized numeric customer id.

Before doing any feature engineering or modeling on customer text, we want to **prove** this pattern with counts (not vibes), then decide whether the leading mention is meaningful signal for an urgency model or just structural addressing noise.


In [10]:
import re

# Matches a leading mention: optional dot, optional whitespace, @, then an alnum/underscore token.
# Captures just the token (no @) for handle-vs-id classification.
LEADING_MENTION_RE = re.compile(r"^\s*\.?@(\w+)")

def first_leading_mention_token(text: str) -> str | None:
    m = LEADING_MENTION_RE.match(text)
    return m.group(1) if m else None

def classify_token(tok) -> str:
    if tok is None or (isinstance(tok, float) and pd.isna(tok)):
        return "no_mention"
    if tok.isdigit():
        return "numeric_id"
    return "handle"


### 9a. Customer tweets — does the `@CompanyHandle` prefix dominate?

In [11]:
customer_tokens = customer["text"].map(first_leading_mention_token)
customer_kind   = customer_tokens.map(classify_token)

n_with = customer_tokens.notna().sum()
n_total = len(customer)
print(f"customer tweets with a leading @-mention: {n_with:,} / {n_total:,} ({n_with / n_total:.1%})")
print()
print("breakdown of leading-token kind (all customer tweets):")
print(customer_kind.value_counts(dropna=False))
print()
print("among customer tweets that DO have a leading mention:")
print(customer_kind[customer_kind != "no_mention"].value_counts(normalize=True).round(4))


customer tweets with a leading @-mention: 1,224,811 / 1,537,843 (79.6%)

breakdown of leading-token kind (all customer tweets):
text
handle        941910
no_mention    313032
numeric_id    282901
Name: count, dtype: int64

among customer tweets that DO have a leading mention:
text
handle        0.769
numeric_id    0.231
Name: proportion, dtype: float64


### 9b. Support replies — does the `@123456` numeric-id prefix dominate?

In [12]:
support_tokens = support["text"].map(first_leading_mention_token)
support_kind   = support_tokens.map(classify_token)

n_support_with = support_tokens.notna().sum()
print(f"support replies with a leading @-mention: {n_support_with:,} / {len(support):,} ({n_support_with / len(support):.1%})")
print()
print("breakdown of leading-token kind (all support replies):")
print(support_kind.value_counts(dropna=False))
print()
print("among support replies that DO have a leading mention:")
print(support_kind[support_kind != "no_mention"].value_counts(normalize=True).round(4))


support replies with a leading @-mention: 1,269,835 / 1,273,931 (99.7%)

breakdown of leading-token kind (all support replies):
text
numeric_id    1269225
no_mention       4096
handle            610
Name: count, dtype: int64

among support replies that DO have a leading mention:
text
numeric_id    0.9995
handle        0.0005
Name: proportion, dtype: float64


### 9c. Examples of each pattern

In [13]:
def show(df_, mask, header, n=5):
    print(f"--- {header} ---")
    for t in df_.loc[mask, "text"].head(n).tolist():
        print("-", t[:160])
    print()

show(customer, customer_kind == "handle",     "customer leading @handle (expected: brand handles)")
show(customer, customer_kind == "numeric_id", "customer leading @numeric_id (less common for customers)")
show(support,  support_kind  == "numeric_id", "support leading @numeric_id (expected: anonymized customer ids)")
show(support,  support_kind  == "handle",     "support leading @handle (less common for support)")


--- customer leading @handle (expected: brand handles) ---
- @sprintcare and how do you propose we do that
- @sprintcare I have sent several private messages and no one is responding as usual
- @sprintcare I did.
- @sprintcare is the worst customer service
- @sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯

--- customer leading @numeric_id (less common for customers) ---


- @115714 y’all lie about your “great” connection. 5 bars LTE, still won’t load something. Smh.
- @115714 whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have never in the 4 years I've tried https://t.co/0G98Rt
- @115722 MD. And this was sent to the wrong address https://t.co/dMQ1WZXJOk
- @115721 Hello Duke, Do you have a copy of your bill? What state are your services located in?
^NHP
- @115722 Nobody can find my account or number. I walked out of a store with this. I've explained that they can find my acct via my devices serial #..

--- support leading @numeric_id (expected: anonymized customer ids) ---


- @115712 I understand. I would like to assist you. We would need to get you into a private secured link to further assist.
- @115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.
- @115712 Can you please send us a private message, so that I can gain further details about your account?
- @115713 This is saddening to hear. Please shoot us a DM, so that we can look into this for you. -KC
- @115713 We understand your concerns and we'd like for you to please send us a Direct Message, so that we can further assist you. -AA

--- support leading @handle (less common for support) ---
- @Tesco @117242 @117246 @117241 @sainsburys @117248 @AldiUK @117249 @Morrisons @117250 @117251 No.. C...Can it be?? TESCO... FROM OUTTA NOWHERE 🐍 https://t.co/54
- @Tesco @117242 @sainsburys @117248 @AldiUK @117260 @117249 @117246 @Morrisons @117250 @117251 We're keeping hold of this for now if that's okay :) https://t.co/
- @nationalrailenq @

## 10. Cleaning decision: build `text_raw` and `text_clean_ml`

The leading mention is structural addressing (who you're tweeting *at*), not content about the customer's issue. For an urgency model the brand handle is uninformative — every Sprint customer says `@sprintcare ...`, every Verizon customer says `@VerizonSupport ...`, and the model would just learn the brand. We will:

- Keep `text_raw` exactly as-is (so we never lose the original surface form).
- Build `text_clean_ml` by stripping **only the first leading mention** (and any trailing whitespace it leaves).

We deliberately do **not** strip *all* mentions — a tweet like `@VerizonSupport @115725 my account is locked` becomes `@115725 my account is locked` after first-mention stripping. The remaining `@115725` is a downstream/embedded mention which may carry context (a CC) and is a separate decision for a later slice.


In [14]:
LEADING_MENTION_STRIP_RE = re.compile(r"^\s*\.?@\w+\s*")

def strip_first_leading_mention(text: str) -> str:
    return LEADING_MENTION_STRIP_RE.sub("", text, count=1).strip()

# Sanity check on representative shapes
for t in [
    "@sprintcare and how do you propose we do that",
    ".@VerizonSupport @115725 service is down",
    "@115712 I understand. I would like to assist you.",
    "no leading mention here, just text",
    "@OnlyAHandle",
]:
    print(f"RAW  : {t!r}")
    print(f"CLEAN: {strip_first_leading_mention(t)!r}")
    print()


RAW  : '@sprintcare and how do you propose we do that'
CLEAN: 'and how do you propose we do that'

RAW  : '.@VerizonSupport @115725 service is down'
CLEAN: '@115725 service is down'

RAW  : '@115712 I understand. I would like to assist you.'
CLEAN: 'I understand. I would like to assist you.'

RAW  : 'no leading mention here, just text'
CLEAN: 'no leading mention here, just text'

RAW  : '@OnlyAHandle'
CLEAN: ''



In [15]:
customer["text_raw"]      = customer["text"]
customer["text_clean_ml"] = customer["text"].map(strip_first_leading_mention)

changed       = (customer["text_raw"] != customer["text_clean_ml"]).sum()
empty_after   = (customer["text_clean_ml"] == "").sum()

print(f"customer tweets changed by first-mention strip : {changed:,} ({changed / len(customer):.1%})")
print(f"customer tweets that become empty after strip  : {empty_after:,} ({empty_after / len(customer):.2%})")


customer tweets changed by first-mention strip : 1,224,811 (79.6%)
customer tweets that become empty after strip  : 1,105 (0.07%)


### Before / after preview

In [16]:
preview = customer[customer["text_raw"] != customer["text_clean_ml"]].head(8)
for _, row in preview.iterrows():
    print("RAW  :", row["text_raw"][:140])
    print("CLEAN:", row["text_clean_ml"][:140])
    print()


RAW  : @sprintcare and how do you propose we do that
CLEAN: and how do you propose we do that

RAW  : @sprintcare I have sent several private messages and no one is responding as usual
CLEAN: I have sent several private messages and no one is responding as usual

RAW  : @sprintcare I did.
CLEAN: I did.

RAW  : @sprintcare is the worst customer service
CLEAN: is the worst customer service

RAW  : @sprintcare You gonna magically change your connectivity for me and my whole family ? 🤥 💯
CLEAN: You gonna magically change your connectivity for me and my whole family ? 🤥 💯

RAW  : @sprintcare Since I signed up with you....Since day 1
CLEAN: Since I signed up with you....Since day 1

RAW  : @115714 y’all lie about your “great” connection. 5 bars LTE, still won’t load something. Smh.
CLEAN: y’all lie about your “great” connection. 5 bars LTE, still won’t load something. Smh.

RAW  : @115714 whenever I contact customer support, they tell me I have shortcode enabled on my account, but I have nev

## 11. Decision

**Decision: strip the first leading `@`-mention for the ML text view, keep the raw text alongside.**

- The `@CompanyHandle` (customer side) and `@123456` (support side) prefix patterns are confirmed by the counts above — both dominate their respective subsets, so this is a real structural artifact, not a fringe case.
- For an urgency classifier on customer tweets, the leading brand handle is constant within each brand and adds no within-brand signal about whether the customer is upset, blocked, or delighted. Leaving it in lets the model trivially overfit to brand frequency.
- We keep `text_raw` so nothing is lost — any downstream task that *does* care about the addressed brand (e.g., per-brand routing, brand-aware features) can still recover it.

**Why this is a cleaning decision, not a direct urgency feature.** "Has a leading mention" is true for ~all customer tweets in this dataset, so it has near-zero variance and would be useless as a model input. The interesting signal it carries (which brand) is a *categorical* feature for a separate slice if we want it — and it's recoverable from `text_raw` at any time. So the right place for this work is the cleaning layer, not the feature layer.

**Out of scope for this mini-slice (intentional):**
- Stripping multiple/embedded mentions
- Handling URLs, emojis, hashtags, HTML entities (`&gt;`, `&amp;`)
- Lowercasing or any tokenization
- Computing engineered features or training any model

Those belong to the next slice.


## 12. Save cleaned text views to `data/processed/`

Small focused CSV with just `tweet_id` and the two text views, so the next slice can join it back to the weak-labeled set without re-running this analysis.


In [17]:
text_views_path = PROCESSED_DIR / "customer_tickets_text_views_v1.csv"
customer[["tweet_id", "text_raw", "text_clean_ml"]].to_csv(text_views_path, index=False)
print("wrote:", text_views_path)
print("rows :", len(customer))


wrote: C:\Users\Jamal\Documents\bootcamps\AIE\Week 3\Decision-Intelligence-Assistant\data\processed\customer_tickets_text_views_v1.csv
rows : 1537843
